In [ ]:
phase 1 validation

def validate_data(df):
    print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
    print("Missing:\n", df.isnull().sum())
    print("Duplicates:", df.duplicated(subset='mid').sum())
    df['label'] = df['disclosure_type'].apply(lambda x: 0 if x == 'NONE' else 1)
    print("Class distribution:\n", df['label'].value_counts())
    print("Imbalance ratio:", round(df['label'].value_counts()[1] / df['label'].value_counts()[0], 2))
    return df

In [ ]:
phase 2 - preprocessing

import re

def clean_text(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    text = text.lower()
    text = re.sub(r'http\S+', '', text)          # remove URLs
    text = re.sub(r'\S+@\S+', '', text)          # remove emails
    text = re.sub(r'\d+', ' NUM ', text)          # normalize numbers
    text = re.sub(r'[^\w\s]', ' ', text)          # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess(df):
    df = df[df['word_count'] <= 2000].copy()      # remove extreme outliers
    df['text_input'] = (
        df['subject'].fillna('').apply(clean_text) + ' ' +
        df['body_clean'].apply(clean_text)
    )
    df = df[df['text_input'].str.len() > 10]      # remove empty after cleaning
    return df



In [ ]:
phase 2b - nlp preprocessing


DISCLOSURE_PHRASES = [
    'confidential', 'do not forward', 'privileged', 'off the record',
    'not for distribution', 'merger', 'acquisition', 'write-down',
    'off-balance', 'special purpose entity', 'reserve', 'attorney',
    'sec filing', 'earnings', 'settlement', 'salary'
]
MODAL_VERBS = ['must', 'shall', 'should', 'cannot', 'may not', 'required to']

def engineer_features(df):
    t = df['body_clean'].fillna('')
    df['f_word_count']             = t.str.split().str.len()
    df['f_avg_word_len']           = t.str.len() / (df['f_word_count'] + 1)
    df['f_disclosure_hits']        = t.str.lower().apply(
                                        lambda x: sum(1 for p in DISCLOSURE_PHRASES if p in x))
    df['f_modal_count']            = t.str.lower().apply(
                                        lambda x: sum(1 for m in MODAL_VERBS if m in x))
    df['f_caps_ratio']             = t.apply(
                                        lambda x: sum(1 for c in x if c.isupper()) / (len(x)+1))
    df['f_has_dollar']             = t.str.contains(
                                        r'\$|\bUSD\b|\bmillion\b|\bbillion\b', regex=True).astype(int)
    df['f_has_legal_term']         = t.str.lower().str.contains(
                                        'attorney|counsel|litigation|sec|ferc', regex=True).astype(int)
    return df

In [ ]:
phase 3 - vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

HAND_FEATURES = [
    'f_word_count', 'f_avg_word_len', 'f_disclosure_hits',
    'f_modal_count', 'f_caps_ratio', 'f_has_dollar', 'f_has_legal_term'
]

def vectorize(X_train_txt, X_test_txt, X_train_hf, X_test_hf):
    tfidf = TfidfVectorizer(
        max_features=15000, ngram_range=(1, 2),
        min_df=2, sublinear_tf=True
    )
    X_tr_tfidf = tfidf.fit_transform(X_train_txt)
    X_te_tfidf = tfidf.transform(X_test_txt)
    
    # Combine TF-IDF with hand features
    X_tr = hstack([X_tr_tfidf, csr_matrix(X_train_hf.values)])
    X_te = hstack([X_te_tfidf, csr_matrix(X_test_hf.values)])
    return X_tr, X_te, tfidf

In [ ]:
phase 4 - swappable model


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

def get_model(name='lr'):
    if name == 'lr':
        return LogisticRegression(
            max_iter=2000, class_weight='balanced',
            C=1.0, solver='saga', random_state=42
        )
    elif name == 'rf':
        return RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            random_state=42, n_jobs=-1
        )
    elif name == 'svm':
        from sklearn.svm import LinearSVC
        return LinearSVC(class_weight='balanced', max_iter=2000)
    else:
        raise ValueError(f"Unknown model: {name}")

def train_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    return model